# NSL-KDD Exploratory Data Analysis

Quick look at the dataset used by the NIDS: class balance, categorical
cardinality and basic feature statistics. Run `python data/download_data.py`
first so the raw files exist under `../data/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.preprocess import load_raw, add_class_column
from src.schema import CATEGORICAL_COLUMNS, NUMERIC_COLUMNS

train = add_class_column(load_raw('../data/KDDTrain+.txt'))
test = add_class_column(load_raw('../data/KDDTest+.txt'))
print('train:', train.shape, ' test:', test.shape)
train.head()

## Class distribution (5-class mapping)
The dataset is heavily imbalanced: Normal and DOS dominate, R2L and U2R are rare.

In [ ]:
dist = pd.DataFrame({
    'train': train['class'].value_counts(),
    'test': test['class'].value_counts(),
})
print(dist)
dist.plot(kind='bar', figsize=(8,4), title='Class distribution (train vs test)');

## Categorical cardinality
`service` has ~70 values, driving most of the one-hot dimensionality.

In [ ]:
for c in CATEGORICAL_COLUMNS:
    print(f'{c:15s} unique={train[c].nunique()}')

print('\nNumeric feature summary:')
train[NUMERIC_COLUMNS].describe().T[['mean','std','min','max']].head(15)

## Correlation of a few traffic features with the attack flag

In [ ]:
tmp = train.copy()
tmp['is_attack'] = (tmp['class'] != 'Normal').astype(int)
cols = ['serror_rate','srv_serror_rate','same_srv_rate','diff_srv_rate',
        'dst_host_serror_rate','count','is_attack']
tmp[cols].corr()['is_attack'].sort_values(ascending=False)